# 2. Which process effects can the ensemble measure?

Four tests of the process model on the CMIP6 land feedbacks, from the cheapest to the most direct:

1. **Budget**: how much of the spread between models do the flags explain?
2. **Power**: how many models would be needed to detect an effect of a given size?
3. **Synthetic recovery**: if known effects are planted in pseudo-data on the real flags, are they found?
4. **Leave one model out**: if each model in turn plays the real world, is it predicted better than by
   the raw ensemble?

The first two are a few lines of linear algebra and run instantly. The last two need many fits, which
`scripts/precompute.py` has cached; `REFIT = True` runs a single live fit for the synthetic test.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tcre_evidence import process_tests as pt          # the development code's process model, and sampling
from tcre_evidence.plotting import style, SERIES
from tcre_evidence.paths import DATA, RESULTS

style()
pd.set_option("display.precision", 2)
CACHE = RESULTS / "precomputed" / "cmip6"   # written by scripts/precompute.py
REFIT = False                               # True: sample live instead of reading the cache (minutes per fit)

values = pd.read_csv(DATA / "examples" / "cmip6_land_feedbacks.csv", index_col="model")
flags = pd.read_csv(DATA / "examples" / "cmip6_process_flags.csv", index_col="model").loc[values.index]
family = flags.pop("family")
PROCESSES = list(flags.columns)

# the quantity to analyse; for land beta use the second line instead
Q, KIND, LABEL = "gamma_land_coubgc", "additive", "γ_L (PgC/K)"
# Q, KIND, LABEL = "beta_land_coubgc", "multiplicative", "β_L (PgC/ppm)"

## 1. Budget: a least-squares view of the flags

Regress the quantity (its logarithm for β_L) on the flags. `r2` is the share of the spread between models
the flags explain; `loo_q2` is that share when each model is predicted from a regression on the others.
A negative `loo_q2` means the flags predict a new model worse than the ensemble mean.

In [ ]:
y = np.log(values[Q].values) if KIND == "multiplicative" else values[Q].values
X = np.column_stack([np.ones(len(y)), flags.values])          # intercept + one column per flag
coef, *_ = np.linalg.lstsq(X, y, rcond=None)
resid = y - X @ coef
s = np.sqrt(resid @ resid / (len(y) - X.shape[1]))           # scatter the flags leave unexplained

loo = [y[i] - X[i] @ np.linalg.lstsq(np.delete(X, i, 0), np.delete(y, i), rcond=None)[0] for i in range(len(y))]
ss = np.sum((y - y.mean()) ** 2)
print(f"r2 = {1 - resid @ resid / ss:.2f},  loo_q2 = {1 - np.sum(np.square(loo)) / ss:.2f},  "
      f"unexplained scatter = {s:.3g} (total spread {y.std(ddof=1):.3g})")

## 2. Power: how many models would be needed?

The standard error of each flag's effect is the unexplained scatter times √[(XᵀX)⁻¹]ₚₚ. Adding models with
the same flag mix shrinks it as 1/√n. An effect is detected at two standard errors when se ≤ effect / 2.
The effect sizes here are half the spread between models (an η of 0.75 for β_L); change `EFFECT` to try
others.

In [ ]:
EFFECT = {p: (abs(np.log(0.75)) if KIND == "multiplicative" else 0.5 * y.std(ddof=1)) for p in PROCESSES}

se_now = s * np.sqrt(np.diag(np.linalg.inv(X.T @ X)))[1:]
copies_needed = np.ceil((2 * se_now / np.array(list(EFFECT.values()))) ** 2)   # se scales as 1/sqrt(copies)
pd.DataFrame({"effect": EFFECT, "estimate now": coef[1:], "se now": se_now,
              "models needed (same mix)": copies_needed * len(y)}, index=PROCESSES)

`pt.detection_power` does the same search, and also for a balanced design in which each flag is set for
half the models at random (the numbers quoted in the note come from it).

In [ ]:
pt.detection_power(pd.Series(y, index=values.index), flags, EFFECT)

## 3. Synthetic recovery

Make a pseudo-ensemble on the real flags with effects you choose and the same scatter as the real data,
fit the process model, and see whether the effects come back. Edit `TRUTH` and set `REFIT = True` for a live
fit (a few minutes).

In [ ]:
if KIND == "additive":
    TRUTH = {"nitrogen": 10.0, "permafrost": 0.0, "fire": 0.0, "veg": 0.0}    # delta, PgC/K
else:
    TRUTH = {"nitrogen": 0.73, "permafrost": 1.0, "fire": 1.0, "veg": 1.0}    # eta factors (1 = no effect)

baseline = np.exp(y.mean()) if KIND == "multiplicative" else y.mean()
fake = pt.synthetic_ensemble(flags, KIND, baseline, TRUTH, scatter_sd=s, rng=np.random.default_rng(1))
fake.round(2).to_frame("pseudo-data").T

In [ ]:
if REFIT:
    idata, diag = pt.fit(pt.build(fake, flags, pt.default_priors(KIND, PROCESSES)),
                         sampler=dict(draws=500, tune=500, chains=2))
    est = idata.posterior[pt.effect_name(KIND)]
    print(diag)
    print(pd.DataFrame({"truth": TRUTH, "estimate": est.mean(("chain", "draw")).values,
                        "posterior sd": est.std(("chain", "draw")).values}, index=PROCESSES))
else:
    print("REFIT is False: the cached replicates below show the same test, repeated 12 times per scenario.")

The cache holds 12 pseudo-ensembles in each of two scenarios: every process has an effect of the size
used above, or no process has any effect. An identified effect follows the truth (the dashed 1:1 line);
one that is not identified stays near its prior median (dotted) whichever scenario is true.

In [ ]:
on = pd.read_csv(CACHE / Q / "synthetic_effects_summary.csv", index_col="term")
off = pd.read_csv(CACHE / Q / "synthetic_no_effects_summary.csv", index_col="term")
prior_median = (pd.read_csv(CACHE / Q / "prior_vs_posterior.csv")
                .assign(process=lambda t: t.term.str.extract(r"\[(.*)\]")[0]).set_index("process").prior_median)

fig, axs = plt.subplots(1, len(PROCESSES), figsize=(10, 2.6))
for ax, p in zip(axs, PROCESSES):
    truth = [off.loc[p, "truth"], on.loc[p, "truth"]]
    est = [off.loc[p, "mean_estimate"], on.loc[p, "mean_estimate"]]
    sd = [off.loc[p, "post_sd"], on.loc[p, "post_sd"]]
    lim = [min(truth + est) - max(sd), max(truth + est) + max(sd)]
    ax.plot(lim, lim, "--", color="0.5", lw=0.8)
    ax.axhline(prior_median[p], ls=":", color=SERIES[1])
    ax.errorbar(truth, est, yerr=sd, fmt="o", color=SERIES[0])
    ax.set(title=p, xlabel="true effect")
axs[0].set_ylabel("estimate (± posterior sd)")
fig.tight_layout()

`tracking` sums this up: how far the estimate moves when the truth moves (1 = follows the truth, 0 =
ignores it, so the answer comes from the prior).

In [ ]:
processes = [p for p in on.index if p != "real_world"]
pd.DataFrame({"truth (no effects / effects)": [f"{off.loc[p, 'truth']:.3g} / {on.loc[p, 'truth']:.3g}" for p in processes],
              "estimate": [f"{off.loc[p, 'mean_estimate']:.3g} / {on.loc[p, 'mean_estimate']:.3g}" for p in processes],
              "tracking": [(on.loc[p, "mean_estimate"] - off.loc[p, "mean_estimate"])
                           / (on.loc[p, "truth"] - off.loc[p, "truth"]) for p in processes]}, index=processes)

## 4. Leave one model out

Each model (with its family, where the flags table has one) is removed in turn, the process model is
refitted, and three predictions of the held-out value are compared with the truth:

* **raw ensemble**: the training models' mean and spread;
* **all processes on**: the real-world estimate, which is what the configuration reports for the real world;
* **own flags**: the prediction for a model with the held-out model's flags.

The real-world estimate is not expected to match a model that lacks some processes. The point is that
nothing in the ensemble can check the adjustment it makes.

In [ ]:
scores = pd.read_csv(CACHE / Q / "loo_scores.csv")
models = list(dict.fromkeys(scores.model))

fig, ax = plt.subplots(figsize=(9, 3))
for j, method in enumerate(["raw_ensemble", "all_processes_on", "own_flags"]):
    g = scores[scores.method == method].set_index("model").loc[models]
    x = np.arange(len(models)) + 0.25 * (j - 1)
    ax.vlines(x, g.p05 - g.truth, g.p95 - g.truth, color=SERIES[j], alpha=0.6)
    ax.plot(x, g["mean"] - g.truth, "o", color=SERIES[j], label=method.replace("_", " "))
ax.axhline(0, color="0.5", lw=0.8)
ax.set_xticks(range(len(models)), models, rotation=40, ha="right")
ax.set_ylabel(f"estimate − held-out value")
ax.legend(fontsize=7, ncol=3);

CRPS (continuous ranked probability score) rewards predictions that are both close and appropriately
spread; skill below 0 means worse than the raw ensemble. Coverage is the share of held-out values inside
the 66 % and 90 % intervals.

In [ ]:
summary = scores.assign(error=scores["mean"] - scores.truth).groupby("method").agg(
    crps=("crps", "mean"), bias=("error", "mean"), coverage_66=("cov66", "mean"), coverage_90=("cov90", "mean"))
summary["crps_skill"] = 1 - summary.crps / summary.loc["raw_ensemble", "crps"]
summary

## Both feedbacks at a glance

The same three numbers per process for β_L and γ_L, read from the cache.

In [ ]:
rows = []
for q in ["beta_land_coubgc", "gamma_land_coubgc"]:
    ev = pt.evidence_table(*(pd.read_csv(CACHE / q / f) for f in ["prior_vs_posterior.csv", "synthetic_effects_summary.csv",
                                                                    "synthetic_no_effects_summary.csv", "power.csv"]))
    rows.append(ev.assign(quantity=q))
pd.concat(rows).set_index(["quantity", "process"])

A process effect is supported by the ensemble when the power test says the ensemble is large enough,
the synthetic test tracks the truth, the posterior differs from the prior, and the flagged models do not all
come from one family. Where these fail, the effect in the result comes from the prior and is better stated
as a judgement. A failed test does not show that a process is unimportant, only that this ensemble cannot
measure it; calibration can also hide a real effect (`scripts/m9_tuning_selection.py`).